In [1]:
# Preparacion

import sys
sys.path.append("..")

import pandas as pd
from statsmodels.stats.weightstats import DescrStatsW
from src.data import cargar_enemdu, tasa_ponderada

df = cargar_enemdu()
df["sexo"] = df["p02"].map({1: "Hombre", 2: "Mujer"})
df["zona"] = df["area"].map({1: "Urbana", 2: "Rural"})

In [2]:
# Indicadores por grupo

def indicadores_por_grupo(df, grupo):
    filas = {}
    for valor, sub in df.groupby(grupo):
        c = sub["condact"]
        pea = c.between(1, 8)
        filas[valor] = {
            "Desempleo": tasa_ponderada(sub, c.isin([7, 8]), pea),
            "Empleo adecuado": tasa_ponderada(sub, c == 1, pea),
            "Subempleo": tasa_ponderada(sub, c.isin([2, 3]), pea),
            "PEA (miles)": sub.loc[pea, "fexp"].sum() / 1000,
        }
    return pd.DataFrame(filas).T.round(1)

print(indicadores_por_grupo(df, "sexo"))
print(indicadores_por_grupo(df, "zona"))

        Desempleo  Empleo adecuado  Subempleo  PEA (miles)
Hombre        2.6             39.5       20.0       5109.1
Mujer         4.6             30.2       16.1       3510.7
        Desempleo  Empleo adecuado  Subempleo  PEA (miles)
Rural         1.7             18.0       17.4       2954.0
Urbana        4.2             44.9       18.9       5665.8


In [3]:
# Informalidad

ocupados = df["condact"].between(1, 6)
informal = df["secemp"] == 2      # 2 = Sector Informal

print("Informalidad nacional (% de ocupados):",
      round(tasa_ponderada(df, informal, ocupados), 1))

for g in ["sexo", "zona"]:
    for valor, sub in df[ocupados].groupby(g):
        tasa = tasa_ponderada(sub, sub["secemp"] == 2, sub["fexp"].notna())
        print(f"{g} = {valor}: {tasa:.1f}")

Informalidad nacional (% de ocupados): 53.5
sexo = Hombre: 54.6
sexo = Mujer: 52.0
zona = Rural: 75.7
zona = Urbana: 41.7


In [4]:
# Ingresos por sexo

ing = df[df["ingrl"] > 0]

def resumen_ingreso(sub):
    d = DescrStatsW(sub["ingrl"], weights=sub["fexp"])
    return pd.Series({
        "media": d.mean,
        "mediana": d.quantile([0.5], return_pandas=False)[0],
        "n (sin ponderar)": len(sub),
    })

res = ing.groupby("sexo").apply(resumen_ingreso).round(1)
res["mediana/hombres"] = (res["mediana"] / res.loc["Hombre", "mediana"]).round(3)
res

,media,mediana,n (sin ponderar),mediana/hombres
sexo,,,,
Hombre,516.4,430.0,20699.0,1.000
Mujer,467.6,350.0,14624.0,0.814


In [5]:
# Formalidad

ocupados = df["condact"].between(1, 6)
for codigo, nombre in {1: "Formal", 2: "Informal", 3: "Doméstico", 4: "No clasificado"}.items():
    print(nombre, round(tasa_ponderada(df, df["secemp"] == codigo, ocupados), 1))

Formal 42.2
Informal 53.5
Doméstico 2.4
No clasificado 1.8
